<h1 style="font-size:30px;">Uber Future Ride Price Prediction</h1>

In [ ]:
!pip install geopy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import plotly.express as px
import plotly.graph_objs as go

In [ ]:
df = pd.read_csv(r"C:\Users\LENOVO\Downloads\uber.csv\uber.csv")

In [ ]:
df.head()

In [ ]:
# Display the first few rows of the dataset
df.info()

In [ ]:
# Drop the unnecessary columns
df.drop(columns=['Unnamed: 0', 'key'], inplace=True)

In [ ]:
# Convert pickup_datetime to datetime format
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df.info()

In [ ]:
# Handle missing values by dropping rows with missing values
df.dropna(inplace=True)

In [ ]:
# Check for any other inconsistencies or outliers
summary_stats = df.describe()
summary_stats

In [ ]:
# Remove outliers in fare_amount
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] <= 500)]

In [ ]:
# Define realistic coordinate ranges for New York City
min_longitude, max_longitude = -74.05, -73.75
min_latitude, max_latitude = 40.63, 40.85

In [ ]:
# Remove entries with unrealistic pickup and dropoff coordinates
df = df[(df['pickup_longitude'].between(min_longitude, max_longitude)) &
                      (df['pickup_latitude'].between(min_latitude, max_latitude)) &
                      (df['dropoff_longitude'].between(min_longitude, max_longitude)) &
                      (df['dropoff_latitude'].between(min_latitude, max_latitude))]

In [ ]:
# Remove entries with unrealistic passenger counts
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]
df.info()

In [ ]:
# Check the summary statistics again after cleaning
cleaned_summary_stats = df.describe()
cleaned_summary_stats

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the plotting environment
sns.set(style="whitegrid")
plt.figure(figsize=(14, 10))

# 1. Distribution of fare_amount
plt.subplot(2, 2, 1)
sns.histplot(df['fare_amount'], bins=50, kde=True)
plt.title('Distribution of Fare Amount')
plt.xlabel('Fare Amount ($)')
plt.ylabel('Frequency')

# 2. Distribution of passenger_count
plt.subplot(2, 2, 2)
sns.countplot(x='passenger_count', data=df)
plt.title('Distribution of Passenger Count')
plt.xlabel('Passenger Count')
plt.ylabel('Frequency')

# 3. Pickup and dropoff locations visualization
plt.subplot(2, 2, 3)
sns.scatterplot(x='pickup_longitude', y='pickup_latitude', data=df, s=1, alpha=0.5)
plt.title('Pickup Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.subplot(2, 2, 4)
sns.scatterplot(x='dropoff_longitude', y='dropoff_latitude', data=df, s=1, alpha=0.5)
plt.title('Dropoff Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.tight_layout()
plt.show()

# 4. Relationship between fare_amount and other features
plt.figure(figsize=(14, 6))

# Relationship with passenger_count
plt.subplot(1, 2, 1)
sns.boxplot(x='passenger_count', y='fare_amount', data=df)
plt.title('Fare Amount vs Passenger Count')
plt.xlabel('Passenger Count')
plt.ylabel('Fare Amount ($)')

# Relationship with distance (calculated)
df['distance'] = ((df['pickup_longitude'] - df['dropoff_longitude'])**2 +
                         (df['pickup_latitude'] - df['dropoff_latitude'])**2)**0.5

plt.subplot(1, 2, 2)
sns.scatterplot(x='distance', y='fare_amount', data=df, alpha=0.5)
plt.title('Fare Amount vs Distance')
plt.xlabel('Distance')
plt.ylabel('Fare Amount ($)')

plt.tight_layout()
plt.show()

# 5. Time-based analysis
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek

plt.figure(figsize=(14, 10))

# Fare amount by hour of day
plt.subplot(2, 1, 1)
sns.boxplot(x='pickup_hour', y='fare_amount', data=df)
plt.title('Fare Amount by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Fare Amount ($)')

# Fare amount by day of week
plt.subplot(2, 1, 2)
sns.boxplot(x='pickup_dayofweek', y='fare_amount', data=df)
plt.title('Fare Amount by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Fare Amount ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['distance'] = np.sqrt((df['pickup_longitude'] - df['dropoff_longitude'])**2 +
                                (df['pickup_latitude'] - df['dropoff_latitude'])**2)

In [ ]:
# Select features and target variable
features = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
            'passenger_count', 'pickup_hour', 'pickup_dayofweek', 'distance']
X = df[features]
y = df['fare_amount']

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train and evaluate Linear Regression model
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f'Linear Regression - RMSE: {rmse_lr}, MAE: {mae_lr}, R²: {r2_lr}')

In [ ]:
# Train and evaluate Decision Tree model
model_dt = DecisionTreeRegressor()
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print(f'Decision Tree - RMSE: {rmse_dt}, MAE: {mae_dt}, R²: {r2_dt}')

In [ ]:
# Train and evaluate K-Nearest Neighbors Regressor model
model_knn = KNeighborsRegressor(n_neighbors=5)
model_knn.fit(X_train, y_train)
y_pred_knn = model_knn.predict(X_test)

# Calculate evaluation metrics
rmse_knn = np.sqrt(mean_squared_error(y_test, y_pred_knn))
mae_knn = mean_absolute_error(y_test, y_pred_knn)
r2_knn = r2_score(y_test, y_pred_knn)

print(f'KNN Regressor - RMSE: {rmse_knn}, MAE: {mae_knn}, R²: {r2_knn}')

In [ ]:
# Train and evaluate Gradient Boosting model
model_gb = GradientBoostingRegressor()
model_gb.fit(X_train, y_train)
y_pred_gb = model_gb.predict(X_test)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb = mean_absolute_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f'Gradient Boosting - RMSE: {rmse_gb}, MAE: {mae_gb}, R²: {r2_gb}')

In [ ]:
def plot_actual_vs_predicted(y_test, y_pred, model_name):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=y_test, y=y_pred, mode='markers', name='Predicted',
                             marker=dict(color='rgba(152, 0, 0, .8)')))
    fig.add_trace(go.Scatter(x=y_test, y=y_test, mode='lines', name='Actual',
                             line=dict(color='rgba(0, 152, 0, .8)')))

    fig.update_layout(title=f'Actual vs Predicted: {model_name}',
                      xaxis_title='Actual Values',
                      yaxis_title='Predicted Values',
                      width=800,
                      height=600)
    fig.show()

# Plot for Linear Regression
plot_actual_vs_predicted(y_test, y_pred_lr, 'Linear Regression')

# Plot for Decision Tree
plot_actual_vs_predicted(y_test, y_pred_dt, 'Decision Tree')

# Plot for Random Forest
plot_actual_vs_predicted(y_test, y_pred_knn, 'KNN')

# Plot for Gradient Boosting
plot_actual_vs_predicted(y_test, y_pred_gb, 'Gradient Boosting')